In [1]:
%load_ext autoreload
%autoreload 2

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import joblib
warnings.filterwarnings('ignore')

sys.path.append('../../scripts')
from data_pipeline import load_and_prepare_data
import plots, xgb_scripts, evaluate

In [4]:
# Load the high-performing model and data
models = joblib.load("../../models/xgb_binning_all_freq.pkl")

X_train, X_test, y_train, y_test = load_and_prepare_data(
    data_folder="../../data/04-03-24", 
    cycle_range=(1, 200),
    method="bin_and_split"
)

# Get predictions and performance
y_pred_mean, y_pred_std, all_predictions = xgb_scripts.predict_ensemble(models, X_test)
rmse, r2, mse, mae = evaluate.evaluate_model(y_test, y_pred_mean)

print(f"Model Performance: R² = {r2:.4f}, RMSE = {rmse:.2f}")
print(f"Data shapes: Train {X_train.shape}, Test {X_test.shape}")
print(f"Ensemble size: {len(models)} models")

X_train: (191, 140), y_train: (191,)
X_test: (84, 140), y_test: (84,)
Train capacity range: 1120.0 - 4050.0 mAh
Test capacity range: 1630.0 - 3880.0 mAh
Model Performance: R² = 0.9283, RMSE = 121.45
Data shapes: Train (191, 140), Test (84, 140)
Ensemble size: 10 models


## Model Uncertainty Analysis

Analyzing prediction uncertainty and ensemble agreement patterns.

In [5]:
# Analyze prediction uncertainty patterns
uncertainty_metrics = {
    'mean_uncertainty': np.mean(y_pred_std),
    'max_uncertainty': np.max(y_pred_std),
    'uncertainty_range': np.max(y_pred_std) - np.min(y_pred_std)
}

# Find high and low uncertainty predictions
high_uncertainty_idx = np.argsort(y_pred_std)[-5:]
low_uncertainty_idx = np.argsort(y_pred_std)[:5]

print("Prediction Uncertainty Analysis:")
print(f"Mean uncertainty: {uncertainty_metrics['mean_uncertainty']:.4f}")
print(f"Max uncertainty: {uncertainty_metrics['max_uncertainty']:.4f}")

# Examine what makes predictions uncertain
high_uncertainty_capacities = y_test[high_uncertainty_idx]
low_uncertainty_capacities = y_test[low_uncertainty_idx]

print(f"High uncertainty predictions (capacity range): {high_uncertainty_capacities.min():.0f} - {high_uncertainty_capacities.max():.0f}")
print(f"Low uncertainty predictions (capacity range): {low_uncertainty_capacities.min():.0f} - {low_uncertainty_capacities.max():.0f}")

Prediction Uncertainty Analysis:
Mean uncertainty: 0.0002
Max uncertainty: 0.0005
High uncertainty predictions (capacity range): 3240 - 3680
Low uncertainty predictions (capacity range): 3230 - 3750


## Model Agreement and Diversity Analysis

Understanding how individual models in the ensemble behave.

In [6]:
# Analyze individual model performance and diversity
individual_r2_scores = []
for i, model in enumerate(models):
    pred = model.predict(X_test)
    r2 = evaluate.evaluate_model(y_test, pred)[1]
    individual_r2_scores.append(r2)

individual_r2_scores = np.array(individual_r2_scores)

# Calculate pairwise correlations between model predictions
model_correlations = np.corrcoef(all_predictions)

print("Ensemble Diversity Analysis:")
print(f"Individual model R² range: {individual_r2_scores.min():.4f} - {individual_r2_scores.max():.4f}")
print(f"Individual model R² std: {individual_r2_scores.std():.4f}")
print(f"Mean pairwise correlation: {np.mean(model_correlations[np.triu_indices_from(model_correlations, k=1)]):.4f}")

# Find best and worst individual models
best_model_idx = np.argmax(individual_r2_scores)
worst_model_idx = np.argmin(individual_r2_scores)

print(f"Best individual model R²: {individual_r2_scores[best_model_idx]:.4f}")
print(f"Worst individual model R²: {individual_r2_scores[worst_model_idx]:.4f}")
print(f"Ensemble improvement: {r2 - individual_r2_scores.max():.4f}")

Ensemble Diversity Analysis:
Individual model R² range: 0.9283 - 0.9283
Individual model R² std: 0.0000
Mean pairwise correlation: 1.0000
Best individual model R²: 0.9283
Worst individual model R²: 0.9283
Ensemble improvement: 0.0000


## Error Pattern Analysis

Investigating where and why the model makes errors.

In [7]:
# Analyze prediction errors by capacity range
residuals = y_test - y_pred_mean
abs_errors = np.abs(residuals)

# Bin predictions by capacity range
capacity_bins = pd.qcut(y_test, q=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
error_by_range = pd.DataFrame({
    'capacity_range': capacity_bins,
    'abs_error': abs_errors,
    'relative_error': abs_errors / y_test * 100
})

range_stats = error_by_range.groupby('capacity_range').agg({
    'abs_error': ['mean', 'std', 'max'],
    'relative_error': ['mean', 'std', 'max']
}).round(2)

print("Error Analysis by Capacity Range:")
print(range_stats)

# Find worst predictions
worst_predictions_idx = np.argsort(abs_errors)[-3:]
print(f"\nWorst predictions:")
for i, idx in enumerate(worst_predictions_idx):
    print(f"{i+1}. True: {y_test[idx]:.0f}, Pred: {y_pred_mean[idx]:.0f}, Error: {abs_errors[idx]:.0f}")

Error Analysis by Capacity Range:
               abs_error                 relative_error             
                    mean     std     max           mean   std    max
capacity_range                                                      
Very Low          183.61  137.41  443.37           7.69  6.74  19.19
Low               100.05   81.25  340.37           2.89  2.34   9.89
Medium             11.76   21.94   79.06           0.33  0.61   2.17
High               26.18   46.90  159.59           0.71  1.28   4.34
Very High          26.11   39.29  147.90           0.70  1.05   3.95

Worst predictions:
1. True: 3440, Pred: 3100, Error: 340
2. True: 2780, Pred: 3157, Error: 377
3. True: 2310, Pred: 2753, Error: 443


## Testing Diverse Ensemble Strategy

Let's retrain with the new diverse ensemble approach to see if it creates different models.

In [8]:
# Train a new CV-based ensemble to compare
print("Training new CV-based ensemble...")
diverse_models = xgb_scripts.train_ensemble_model(X_train, y_train)

# Get predictions from CV ensemble
y_pred_diverse_mean, y_pred_diverse_std, all_diverse_predictions = xgb_scripts.predict_ensemble(diverse_models, X_test)
rmse_diverse, r2_diverse, mse_diverse, mae_diverse = evaluate.evaluate_model(y_test, y_pred_diverse_mean)

print(f"\nCV Ensemble Performance: R² = {r2_diverse:.4f}, RMSE = {rmse_diverse:.2f}")
print(f"Original Ensemble Performance: R² = {r2:.4f}, RMSE = {rmse:.2f}")

# Analyze diversity of CV ensemble
diverse_individual_r2 = []
for i, model in enumerate(diverse_models):
    pred = model.predict(X_test)
    r2_score = evaluate.evaluate_model(y_test, pred)[1]
    diverse_individual_r2.append(r2_score)

diverse_individual_r2 = np.array(diverse_individual_r2)
diverse_correlations = np.corrcoef(all_diverse_predictions)

print(f"\nCV Ensemble Diversity Analysis:")
print(f"Individual model R² range: {diverse_individual_r2.min():.4f} - {diverse_individual_r2.max():.4f}")
print(f"Individual model R² std: {diverse_individual_r2.std():.4f}")
print(f"Mean pairwise correlation: {np.mean(diverse_correlations[np.triu_indices_from(diverse_correlations, k=1)]):.4f}")
print(f"Ensemble improvement: {r2_diverse - diverse_individual_r2.max():.4f}")

# Compare uncertainty
diverse_uncertainty = {
    'mean_uncertainty': np.mean(y_pred_diverse_std),
    'max_uncertainty': np.max(y_pred_diverse_std),
}

print(f"\nUncertainty Comparison:")
print(f"Original ensemble mean uncertainty: {uncertainty_metrics['mean_uncertainty']:.4f}")
print(f"CV ensemble mean uncertainty: {diverse_uncertainty['mean_uncertainty']:.4f}")
print(f"CV ensemble max uncertainty: {diverse_uncertainty['max_uncertainty']:.4f}")

Training new CV-based ensemble...

CV Ensemble Performance: R² = 0.9395, RMSE = 111.53
Original Ensemble Performance: R² = 0.9283, RMSE = 121.45

CV Ensemble Diversity Analysis:
Individual model R² range: 0.8598 - 0.9417
Individual model R² std: 0.0274
Mean pairwise correlation: 0.9581
Ensemble improvement: -0.0022

Uncertainty Comparison:
Original ensemble mean uncertainty: 0.0002
CV ensemble mean uncertainty: 55.9096
CV ensemble max uncertainty: 303.8649

CV Ensemble Performance: R² = 0.9395, RMSE = 111.53
Original Ensemble Performance: R² = 0.9283, RMSE = 121.45

CV Ensemble Diversity Analysis:
Individual model R² range: 0.8598 - 0.9417
Individual model R² std: 0.0274
Mean pairwise correlation: 0.9581
Ensemble improvement: -0.0022

Uncertainty Comparison:
Original ensemble mean uncertainty: 0.0002
CV ensemble mean uncertainty: 55.9096
CV ensemble max uncertainty: 303.8649
